# Lab: Assistants API Agent

## Lab Overview

This lab walks you through building production-grade AI agents using **LangChain** and **Azure OpenAI** with a deployed **GPT-4o** model. Rather than making single-turn API calls, you will construct agents that reason over tasks, decide which tools to invoke, observe results, and continue reasoning until a goal is reached. This is the paradigm that underlies real-world automation pipelines in finance, healthcare, and enterprise operations.

The lab is structured around the **ReAct** (Reason + Act) pattern, which is the foundation of modern LLM agents. You will see how LangChain wraps Azure OpenAI endpoints into a clean agent abstraction, how tools are defined and registered, and how the agent loop drives multi-step execution without human intervention at each step.


## What You Will Learn

By completing this lab you will be able to:

1. Understand the theoretical foundations of LLM agents: the ReAct loop, tool calling, and chain-of-thought reasoning.
2. Configure LangChain to connect to an Azure OpenAI endpoint using your deployed GPT-4o model.
3. Define custom tools and register them with a LangChain agent.
4. Build and run ReAct agents using `langgraph.prebuilt.create_react_agent`.
5. Add conversational memory so the agent maintains context across turns.
6. Build domain-specific agents rooted in finance and healthcare scenarios.
7. Inspect and interpret agent traces to debug and optimise agent behaviour.

## Prerequisites

- An **Azure OpenAI resource** provisioned in your Azure subscription.
- A **GPT-4o deployment** with its deployment name, endpoint URL, and API key.
- Python 3.10 or later (3.12 recommended).
- Familiarity with Python, REST concepts, and a basic understanding of large language models.
- A Jupyter-compatible environment (JupyterLab, VS Code Jupyter extension, or Azure Machine Learning Notebooks).

## Section 1 — Theoretical Foundations

### 1.1 What Is an LLM Agent?

A large language model on its own is a stateless function: it receives a prompt and returns a completion. An **agent** augments that function with the ability to:

- Decide what action to take next (reason).
- Execute that action against the real world (act).
- Observe the result and update its reasoning state.
- Repeat until the goal is satisfied.

This loop is fundamentally different from a single API call. In a financial context, think of the difference between asking a model "what is the current yield on a 10-year Treasury?" (the model guesses or hallucinates) versus giving an agent a live data tool so it can actually fetch the yield and then reason about portfolio implications.

### 1.2 The ReAct Framework

ReAct (Yao et al., 2022) is the dominant prompting strategy for agents. The model interleaves **Thought** (chain-of-thought reasoning), **Action** (tool invocation), and **Observation** (tool result) steps before producing a final answer.

A ReAct trace for a healthcare query:

```
Thought: The user wants to know whether metformin interacts with ibuprofen. I should look this up.
Action: check_drug_interaction({"drug_a": "metformin", "drug_b": "ibuprofen"})
Observation: Ibuprofen can reduce renal clearance of metformin, increasing plasma levels.
Thought: I now have the answer. I should present this with a clinical caution.
Final Answer: There is a moderate interaction between metformin and ibuprofen ...
```

The key insight is that the model never pretends to know things it cannot know. It uses tools to ground its answers in real data.

### 1.3 LangChain and LangGraph Architecture

The LangChain ecosystem after 1.0 has two tiers:

**LangGraph (`langgraph`)** — A graph-based runtime for fine-grained, controllable agent construction. The `create_react_agent` function from `langgraph.prebuilt` compiles a `StateGraph` with two nodes (`agent` and `tools`) connected by a conditional edge. The edge checks whether the model's last message contained tool calls. If yes, it routes to `tools`, executes them, and feeds results back to `agent`. If no, execution terminates. This is the approach this lab uses.

**LangChain (`langchain`)** — Higher-level abstractions for rapid agent development. The `create_agent` function from `langchain.agents` builds on LangGraph internally and adds middleware hooks for human-in-the-loop, summarisation, and PII redaction.

### 1.4 Why the Version Mismatch Causes an ImportError

The `langgraph-prebuilt` sub-package imports `_DirectlyInjectedToolArg` from `langchain_core.tools.base`. This symbol was introduced in `langchain-core>=1.0`. If you have an older `langchain-core` (0.3.x) installed alongside a newer `langgraph-prebuilt`, the import fails.

The fix is always the same: install all packages together in a single `pip install` command without version pins, so pip can resolve a mutually compatible set. Never pin `langchain-core` independently from `langgraph`.

### 1.5 Azure OpenAI vs. OpenAI Direct

Key differences from calling `api.openai.com` directly:

- You authenticate with your Azure resource endpoint and API key (or Azure Active Directory).
- You address models by **deployment name**, not model name.
- Data stays within your Azure tenant, satisfying healthcare (HIPAA) and financial (SOC 2, GDPR) compliance requirements.
- The API is versioned. Always check the Azure OpenAI documentation for the current supported `api-version` strings.

## Section 2 — Environment Setup

### Step 2.1 — Install Required Packages

**Why we do not pin versions individually here:**

The `_DirectlyInjectedToolArg ImportError` and similar failures come from pip installing packages with conflicting version requirements. The correct fix is to install everything in one command with `--upgrade`, letting pip's dependency resolver find a coherent set. The `langchain`, `langchain-openai`, and `langgraph` packages all declare their `langchain-core` version bounds as dependencies, so the resolver handles it automatically.

After running this cell, **restart the kernel before continuing**. This ensures the freshly installed packages are loaded cleanly.

In [1]:
# Install all LangChain ecosystem packages in a single command.
# Using --upgrade ensures any older conflicting versions are replaced.
# Do NOT pin langchain-core separately from langgraph — this is the root cause of import errors.
#
# RESTART THE KERNEL after this cell completes.

%pip install \
    langchain \
    langchain-openai \
    langchain-community \
    langgraph \
    openai \
    python-dotenv \
    tiktoken \
    grandalf \
    --upgrade --quiet

Note: you may need to restart the kernel to use updated packages.


> **Restart the kernel now** (Kernel > Restart Kernel) before running any cells below this line.

### Step 2.2 — Verify Installed Versions

**Why this step matters:** Recording exact versions makes the environment reproducible and gives you a reference point if a future upgrade introduces a breaking change. Save this output alongside your notebook.

In [2]:
import importlib
import sys

print(f"Python: {sys.version}")
print()

packages = [
    "langchain_core",
    "langchain",
    "langchain_community",
    "openai",
    "tiktoken",
]

print("Installed versions (record these for reproducibility):")
print("-" * 55)
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:<30} {getattr(mod, '__version__', 'unknown')}")
    except ImportError:
        print(f"{pkg:<30} NOT INSTALLED")

print()
print("Critical import checks:")
print("-" * 55)

checks = [
    ("langchain_openai",          "AzureChatOpenAI"),
    ("langchain_core.tools",      "tool"),
    ("langchain_core.messages",   "HumanMessage"),
    ("langgraph.prebuilt",        "create_react_agent"),
    ("langgraph.checkpoint.memory", "MemorySaver"),
]

all_ok = True
for module_path, symbol in checks:
    try:
        mod = importlib.import_module(module_path)
        getattr(mod, symbol)
        print(f"  OK   from {module_path} import {symbol}")
    except (ImportError, AttributeError) as exc:
        print(f"  FAIL from {module_path} import {symbol}")
        print(f"       {exc}")
        all_ok = False

print()
if all_ok:
    print("All critical imports healthy. Proceed to the next section.")
else:
    print("One or more imports failed.")
    print("Fix: re-run the installation cell, restart the kernel, then run this cell again.")

Python: 3.12.1 (tags/v3.12.1:2305ca5, Dec  7 2023, 22:03:25) [MSC v.1937 64 bit (AMD64)]

Installed versions (record these for reproducibility):
-------------------------------------------------------
langchain_core                 1.2.18
langchain                      1.2.12
langchain_community            0.4.1
openai                         2.26.0
tiktoken                       0.12.0

Critical import checks:
-------------------------------------------------------
  OK   from langchain_openai import AzureChatOpenAI
  OK   from langchain_core.tools import tool
  OK   from langchain_core.messages import HumanMessage
  OK   from langgraph.prebuilt import create_react_agent
  OK   from langgraph.checkpoint.memory import MemorySaver

All critical imports healthy. Proceed to the next section.


### Step 2.3 — Configure Credentials

**Why this step matters:** Hardcoding API keys into notebooks is a security risk. Even if the notebook is never shared, secrets committed to version control can be leaked. The best practice is to read credentials from environment variables or a `.env` file that lives outside your repository.

Create a file named `.env` in the same directory as this notebook with the following content (replacing the placeholder values with your real Azure OpenAI values). The below code will do that.

```
AZURE_OPENAI_API_KEY=your_azure_openai_api_key_here
AZURE_OPENAI_ENDPOINT=https://your-resource-name.openai.azure.com/
AZURE_OPENAI_DEPLOYMENT_NAME=gpt-4o
AZURE_OPENAI_API_VERSION=2025-01-01-preview
```

Then run the cell below to load those values into the process environment.

In [3]:
from pathlib import Path

env_content = """
AZURE_OPENAI_ENDPOINT=https://<TBD>.cognitiveservices.azure.com
AZURE_OPENAI_API_KEY=
AZURE_OPENAI_DEPLOYMENT_NAME=gpt-4o
AZURE_OPENAI_API_VERSION=2024-12-01-preview
"""

# Create .env file in current directory
env_path = Path(".env")
env_path.write_text(env_content.strip())

print(".env file created successfully.")
print(env_path.resolve())

.env file created successfully.
\\260302aaibatch.file.core.windows.net\20260302acclaixlabfiles\Day9\.env


In [4]:
import os
from dotenv import load_dotenv

# Load credentials from .env into the process environment.
# override=True means .env values win over existing shell environment variables.
load_dotenv(override=True)

required_vars = [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_DEPLOYMENT_NAME",
    "AZURE_OPENAI_API_VERSION",
]

missing = [v for v in required_vars if not os.getenv(v)]
if missing:
    raise EnvironmentError(
        f"Missing environment variables: {missing}. "
        "Populate your .env file and re-run this cell."
    )

print("All required environment variables loaded.")
print(f"  Endpoint:    {os.getenv('AZURE_OPENAI_ENDPOINT')}")
print(f"  Deployment:  {os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')}")
print(f"  API Version: {os.getenv('AZURE_OPENAI_API_VERSION')}")

All required environment variables loaded.
  Endpoint:    https://<TBD>.cognitiveservices.azure.com
  Deployment:  gpt-4o
  API Version: 2024-12-01-preview



## Section 3 — Connecting LangChain to Azure OpenAI

### Theory: The Chat Model Abstraction

LangChain separates agent logic from the model backend through a `BaseChatModel` interface. `AzureChatOpenAI` is one implementation. You can swap in a different model by changing this one object while keeping all agent code identical — essential for enterprise systems where the model changes due to cost, capability, or compliance requirements.

`temperature=0` produces deterministic outputs, which is almost always what you want for agent decision-making. Higher temperatures are appropriate for creative tasks, not for reasoning about which tool to call next.

### Step 3.1 — Instantiate the Chat Model

In [5]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature=0,      # deterministic for agent decision-making
    max_tokens=4096,    # generous budget for multi-step reasoning chains
)

print(f"Model class: {type(llm).__name__}")
print("AzureChatOpenAI object created successfully.")

Model class: AzureChatOpenAI
AzureChatOpenAI object created successfully.


### Step 3.2 — Verify Connectivity

**Why this step matters:** Before building agents, confirm the endpoint is reachable and credentials are valid. A direct call here isolates connectivity from agent logic — authentication issues are much easier to diagnose without the agent loop involved.

In [6]:
from langchain_core.messages import HumanMessage

# A direct, non-agent call equivalent to a simple chat completion request.
test_response = llm.invoke([
    HumanMessage(
        content="In one sentence, what is the primary purpose of Basel III capital requirements?"
    )
])

print("Model response:")
print(test_response.content)

Model response:
The primary purpose of Basel III capital requirements is to strengthen the regulation, supervision, and risk management of banks by ensuring they maintain adequate capital to absorb shocks and reduce the risk of financial system instability.


## Section 4 — Defining Tools

### Theory: What Makes a Good Tool?

A tool is a Python function that the agent calls to interact with the world outside the model context window. The model never sees raw code — it sees the tool's **name** and **description** and uses those to decide whether the tool is relevant for a given step.

Writing good tool descriptions is like writing good API documentation for a developer who has never seen your codebase. The description must answer: what does this tool do, what inputs does it expect, and when should (and should not) it be used.

GPT-4o uses function-calling natively. When tools are registered, the model emits structured JSON arguments rather than free-text commands — significantly more reliable than older text-parsing approaches.

### Step 4.1 — Finance Domain Tools

In production these would call Bloomberg, Reuters, or internal risk APIs. Mock data is used here so the lab runs without external dependencies.

In [7]:
from langchain_core.tools import tool

@tool
def get_stock_price(ticker: str) -> str:
    """
    Retrieve the current stock price for a given equity ticker symbol.
    Use this tool whenever you need a live or current price for a publicly traded company.
    Input: ticker — the exchange ticker symbol, e.g. AAPL, MSFT, JPM.
    Returns: a string with the current price in USD.
    """
    mock_prices = {
        "AAPL": "189.45",
        "MSFT": "415.20",
        "JPM":  "198.73",
        "GS":   "452.10",
        "JNJ":  "162.88",
        "PFE":  "29.54",
    }
    ticker = ticker.upper().strip()
    price = mock_prices.get(ticker)
    if price:
        return f"The current price of {ticker} is ${price} USD."
    return f"Ticker {ticker} not found in the data source."


@tool
def calculate_portfolio_value(holdings: str) -> str:
    """
    Calculate the total market value of a portfolio given a comma-separated list of
    TICKER:SHARES pairs. Example input: 'AAPL:100,JPM:50,MSFT:200'.
    Use this tool when the user asks for a portfolio valuation or total position value.
    Returns: a breakdown by position and a total portfolio value in USD.
    """
    mock_prices = {
        "AAPL": 189.45, "MSFT": 415.20, "JPM": 198.73,
        "GS": 452.10,   "JNJ": 162.88,  "PFE": 29.54,
    }
    total = 0.0
    lines = []
    try:
        for pair in holdings.split(","):
            ticker, shares_str = pair.strip().split(":")
            ticker = ticker.strip().upper()
            shares = float(shares_str.strip())
            price = mock_prices.get(ticker, 0.0)
            value = price * shares
            total += value
            lines.append(f"  {ticker}: {shares:.0f} shares x ${price:.2f} = ${value:,.2f}")
    except ValueError as exc:
        return f"Error parsing holdings: {exc}. Expected format: 'TICKER:SHARES,TICKER:SHARES'"
    result = "Portfolio breakdown:\n" + "\n".join(lines)
    result += f"\n\nTotal portfolio value: ${total:,.2f} USD"
    return result


@tool
def get_interest_rate(rate_type: str) -> str:
    """
    Look up a benchmark interest rate or bond yield.
    Supported rate_type values: 'fed_funds', 'sofr', 'us_10yr_treasury', 'us_2yr_treasury'.
    Use this tool when you need current rate or yield data for fixed income analysis,
    duration calculations, or macroeconomic context.
    Returns: the current rate as a percentage string.
    """
    mock_rates = {
        "fed_funds":        "5.25%",
        "sofr":             "5.30%",
        "us_10yr_treasury": "4.48%",
        "us_2yr_treasury":  "4.85%",
    }
    key = rate_type.lower().strip()
    rate = mock_rates.get(key)
    if rate:
        return f"Current {rate_type.replace('_', ' ').title()}: {rate}"
    return (
        f"Rate type '{rate_type}' not recognised. "
        "Choose from: fed_funds, sofr, us_10yr_treasury, us_2yr_treasury."
    )


finance_tools = [get_stock_price, calculate_portfolio_value, get_interest_rate]
print(f"Finance tools registered: {[t.name for t in finance_tools]}")

Finance tools registered: ['get_stock_price', 'calculate_portfolio_value', 'get_interest_rate']


### Step 4.2 — Healthcare Domain Tools

In production these would integrate with Epic FHIR APIs, First Databank, or clinical trial registries.

In [8]:
@tool
def check_drug_interaction(drug_a: str, drug_b: str) -> str:
    """
    Check for a known clinical drug interaction between two medications.
    Use this tool whenever the user asks whether two drugs can be taken together,
    or when assessing medication safety for a patient on multiple prescriptions.
    Input: drug_a and drug_b — the generic names of the two medications.
    Returns: interaction severity and a clinical note.
    """
    interaction_db = {
        frozenset(["warfarin", "aspirin"]): (
            "MAJOR",
            "Concomitant use significantly increases bleeding risk. "
            "Monitor INR closely and consider GI prophylaxis."
        ),
        frozenset(["metformin", "ibuprofen"]): (
            "MODERATE",
            "NSAIDs can reduce renal clearance of metformin, increasing plasma levels "
            "and the risk of lactic acidosis. Monitor renal function."
        ),
        frozenset(["lisinopril", "potassium"]): (
            "MODERATE",
            "ACE inhibitors increase serum potassium. Avoid supplements "
            "unless hypokalaemia is confirmed."
        ),
        frozenset(["atorvastatin", "clarithromycin"]): (
            "MAJOR",
            "CYP3A4 inhibition by clarithromycin markedly elevates atorvastatin levels, "
            "increasing myopathy and rhabdomyolysis risk. Suspend statin during antibiotic course."
        ),
    }
    key = frozenset([drug_a.lower().strip(), drug_b.lower().strip()])
    interaction = interaction_db.get(key)
    if interaction:
        severity, note = interaction
        return f"Interaction severity: {severity}\nClinical note: {note}"
    return (
        f"No major interaction documented between {drug_a} and {drug_b} in this database. "
        "Always consult a current clinical reference or pharmacist."
    )


@tool
def get_clinical_guideline(condition: str) -> str:
    """
    Retrieve a summary of the current clinical management guideline for a given medical condition.
    Use this tool when the user asks about treatment protocols, first-line therapy, or
    evidence-based management recommendations for a specific diagnosis.
    Input: condition — the name of the medical condition.
    Supported values: 'type2_diabetes', 'hypertension', 'atrial_fibrillation'.
    Returns: a structured guideline summary.
    """
    guidelines = {
        "type2_diabetes": (
            "ADA 2024 Standards of Care: First-line: Metformin + lifestyle modification "
            "(target HbA1c < 7.0%). Second-line: GLP-1 receptor agonist (preferred if ASCVD "
            "or weight loss needed) or SGLT-2 inhibitor (preferred if heart failure or CKD). "
            "Third-line: Insulin if HbA1c remains uncontrolled."
        ),
        "hypertension": (
            "ACC/AHA 2017: Target BP < 130/80 mmHg. First-line: ACE inhibitor or ARB, "
            "thiazide diuretic, or calcium channel blocker. "
            "Avoid ACE inhibitor + ARB combination due to adverse renal outcomes."
        ),
        "atrial_fibrillation": (
            "ESC 2020 AF Guidelines: Assess stroke risk using CHA2DS2-VASc score. "
            "Anticoagulate if score >= 2 (male) or >= 3 (female). "
            "Preferred anticoagulants: NOACs over warfarin unless contraindicated."
        ),
    }
    key = condition.lower().strip().replace(" ", "_")
    guideline = guidelines.get(key)
    if guideline:
        return guideline
    return (
        f"No guideline found for '{condition}'. "
        "Supported: type2_diabetes, hypertension, atrial_fibrillation."
    )


healthcare_tools = [check_drug_interaction, get_clinical_guideline]
print(f"Healthcare tools registered: {[t.name for t in healthcare_tools]}")

Healthcare tools registered: ['check_drug_interaction', 'get_clinical_guideline']


## Section 5 — Building the Agent

### Theory: `create_react_agent` and the LangGraph State Machine

`create_react_agent` from `langgraph.prebuilt` compiles a `StateGraph`. The graph has two nodes:

- **`agent` node:** sends the current message list to the model and receives a response.
- **`tools` node:** executes any tool calls present in the model's response and appends the results as `ToolMessage` objects.

A conditional edge after the `agent` node checks `tools_condition`: if the last AIMessage has `tool_calls`, route to `tools`; otherwise, end. This loop continues until the model produces a response with no tool calls.

The `prompt` parameter sets the system message. Keep system prompts focused on persona, scope, and compliance disclaimers. Do not instruct the model on when to use specific tools — the tool descriptions handle that.

### Step 5.1 — Finance Agent

In [9]:
from langgraph.prebuilt import create_react_agent

finance_system_prompt = (
    "You are a senior financial analyst assistant with expertise in equity markets, "
    "fixed income, and portfolio management. You have access to real-time market data tools "
    "and use them whenever you need current prices or rates. Never guess or use stale knowledge.\n\n"
    "Present numerical results clearly with appropriate units. When performing calculations, "
    "show your reasoning step by step. Conclude with a brief risk caveat where applicable.\n\n"
    "This system provides informational analysis only and does not constitute investment advice."
)

finance_agent = create_react_agent(
    model=llm,
    tools=finance_tools,
    prompt=finance_system_prompt,
)

print(f"Finance agent type: {type(finance_agent).__name__}")
print("Finance agent created successfully.")

Finance agent type: CompiledStateGraph
Finance agent created successfully.


C:\Users\agenticaiuser\AppData\Local\Temp\ipykernel_10200\1269042749.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  finance_agent = create_react_agent(


In [10]:
def run_agent_and_print(agent, user_query: str) -> str:
    """Invoke an agent, print the message trace, and return the final answer."""
    print(f"Query: {user_query}")
    print("=" * 65)

    result = agent.invoke({"messages": [HumanMessage(content=user_query)]})

    for i, msg in enumerate(result["messages"]):
        msg_type = type(msg).__name__
        print(f"[Step {i + 1}] {msg_type}")
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  Tool called: {tc['name']}")
                print(f"  Arguments:   {tc['args']}")
        elif hasattr(msg, "content") and msg.content:
            preview = str(msg.content)
            print(f"  Content: {preview[:300]}{'...' if len(preview) > 300 else ''}")
        print()

    final = result["messages"][-1].content
    print("=" * 65)
    print("FINAL ANSWER:")
    print(final)
    return final


# Finance Example 1: Multi-tool query combining portfolio value and rate context
run_agent_and_print(
    finance_agent,
    "I hold 200 shares of JPM and 150 shares of GS. What is the current market value of my positions, "
    "and how does the current 10-year Treasury yield affect the attractiveness of these financial stocks?"
)

Query: I hold 200 shares of JPM and 150 shares of GS. What is the current market value of my positions, and how does the current 10-year Treasury yield affect the attractiveness of these financial stocks?
[Step 1] HumanMessage
  Content: I hold 200 shares of JPM and 150 shares of GS. What is the current market value of my positions, and how does the current 10-year Treasury yield affect the attractiveness of these financial stocks?

[Step 2] AIMessage
  Tool called: calculate_portfolio_value
  Arguments:   {'holdings': 'JPM:200,GS:150'}
  Tool called: get_interest_rate
  Arguments:   {'rate_type': 'us_10yr_treasury'}

[Step 3] ToolMessage
  Content: Portfolio breakdown:
  JPM: 200 shares x $198.73 = $39,746.00
  GS: 150 shares x $452.10 = $67,815.00

Total portfolio value: $107,561.00 USD

[Step 4] ToolMessage
  Content: Current Us 10Yr Treasury: 4.48%

[Step 5] AIMessage
  Content: ### Portfolio Valuation
- **JPM (200 shares)**: $198.73 per share × 200 = **$39,746.00**
- **GS (150 sha

"### Portfolio Valuation\n- **JPM (200 shares)**: $198.73 per share × 200 = **$39,746.00**\n- **GS (150 shares)**: $452.10 per share × 150 = **$67,815.00**\n\n**Total Portfolio Value**: **$107,561.00 USD**\n\n### Impact of the 10-Year Treasury Yield (4.48%)\nThe 10-year Treasury yield serves as a benchmark for risk-free returns and influences the cost of capital for financial institutions like JPMorgan Chase (JPM) and Goldman Sachs (GS). Here's how it affects their attractiveness:\n1. **Higher Yield**: A 4.48% yield makes fixed-income investments more competitive compared to equities, potentially reducing demand for financial stocks.\n2. **Net Interest Margin (NIM)**: For banks like JPM, higher yields can improve NIM, as they may charge higher rates on loans while keeping deposit rates relatively lower.\n3. **Valuation Pressure**: Higher yields increase the discount rate used in equity valuation models, potentially lowering the intrinsic value of stocks like JPM and GS.\n\n### Risk Cav

In [11]:
# Finance Example 2: Portfolio calculation + macroeconomic context
run_agent_and_print(
    finance_agent,
    "Calculate the total value of this portfolio: AAPL:100, MSFT:75, JPM:200. "
    "Also tell me the current Fed Funds rate and briefly explain what it means for equity valuations."
)

Query: Calculate the total value of this portfolio: AAPL:100, MSFT:75, JPM:200. Also tell me the current Fed Funds rate and briefly explain what it means for equity valuations.
[Step 1] HumanMessage
  Content: Calculate the total value of this portfolio: AAPL:100, MSFT:75, JPM:200. Also tell me the current Fed Funds rate and briefly explain what it means for equity valuations.

[Step 2] AIMessage
  Tool called: calculate_portfolio_value
  Arguments:   {'holdings': 'AAPL:100,MSFT:75,JPM:200'}
  Tool called: get_interest_rate
  Arguments:   {'rate_type': 'fed_funds'}

[Step 3] ToolMessage
  Content: Portfolio breakdown:
  AAPL: 100 shares x $189.45 = $18,945.00
  MSFT: 75 shares x $415.20 = $31,140.00
  JPM: 200 shares x $198.73 = $39,746.00

Total portfolio value: $89,831.00 USD

[Step 4] ToolMessage
  Content: Current Fed Funds: 5.25%

[Step 5] AIMessage
  Content: ### Portfolio Valuation
- **AAPL**: 100 shares x $189.45 = **$18,945.00**
- **MSFT**: 75 shares x $415.20 = **$31,140.00**

'### Portfolio Valuation\n- **AAPL**: 100 shares x $189.45 = **$18,945.00**\n- **MSFT**: 75 shares x $415.20 = **$31,140.00**\n- **JPM**: 200 shares x $198.73 = **$39,746.00**\n\n**Total Portfolio Value**: **$89,831.00 USD**\n\n---\n\n### Current Fed Funds Rate\nThe current Federal Funds Rate is **5.25%**.\n\n---\n\n### Impact of Fed Funds Rate on Equity Valuations\nThe Federal Funds Rate is the interest rate at which banks lend to each other overnight. It serves as a benchmark for other interest rates, including those for loans and mortgages. A higher Fed Funds Rate generally increases the cost of borrowing, which can reduce corporate profits and consumer spending. This often leads to lower equity valuations as investors demand higher returns to compensate for increased risk and opportunity costs. Conversely, a lower rate tends to support higher equity valuations by reducing borrowing costs and increasing liquidity.\n\n---\n\n**Risk Caveat**: Equity valuations are influenced by multip

### Step 5.2 — Healthcare Agent

In [13]:
healthcare_system_prompt = (
    "You are a clinical decision-support assistant designed to help healthcare professionals "
    "access evidence-based information quickly. You have access to drug interaction databases "
    "and clinical guideline summaries.\n\n"
    "Always use your tools to retrieve current information. Clinical guidelines and interaction "
    "data are updated regularly — do not rely on training data alone.\n\n"
    "Present information clearly and structured. Always include a reminder that clinical decisions "
    "must be made by qualified healthcare professionals in the context of individual patient factors."
)

healthcare_agent = create_react_agent(
    model=llm,
    tools=healthcare_tools,
    prompt=healthcare_system_prompt,
)

print("Healthcare agent created.")

Healthcare agent created.


C:\Users\agenticaiuser\AppData\Local\Temp\ipykernel_10200\674822870.py:11: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  healthcare_agent = create_react_agent(


In [55]:
# Healthcare Example 1: Drug interaction + guideline query
run_agent_and_print(
    healthcare_agent,
    "A patient with type 2 diabetes is on metformin. They have a headache and want to take ibuprofen. "
    "Is this safe, and what does the current guideline say about managing their diabetes?"
)

Query: A patient with type 2 diabetes is on metformin. They have a headache and want to take ibuprofen. Is this safe, and what does the current guideline say about managing their diabetes?
[Step 1] HumanMessage
  Content: A patient with type 2 diabetes is on metformin. They have a headache and want to take ibuprofen. Is this safe, and what does the current guideline say about managing their diabetes?

[Step 2] AIMessage
  Tool called: check_drug_interaction
  Arguments:   {'drug_a': 'metformin', 'drug_b': 'ibuprofen'}
  Tool called: get_clinical_guideline
  Arguments:   {'condition': 'type2_diabetes'}

[Step 3] ToolMessage
  Content: Interaction severity: MODERATE
Clinical note: NSAIDs can reduce renal clearance of metformin, increasing plasma levels and the risk of lactic acidosis. Monitor renal function.

[Step 4] ToolMessage
  Content: ADA 2024 Standards of Care: First-line: Metformin + lifestyle modification (target HbA1c < 7.0%). Second-line: GLP-1 receptor agonist (preferred if A

'### Drug Interaction: Metformin and Ibuprofen\n- **Interaction Severity:** Moderate\n- **Clinical Note:** NSAIDs, including ibuprofen, can reduce renal clearance of metformin, potentially increasing plasma levels and the risk of lactic acidosis. It is advisable to monitor renal function if ibuprofen is used, especially for prolonged periods or in higher doses.\n\n### Current Guideline for Managing Type 2 Diabetes (ADA 2024 Standards of Care)\n1. **First-line Therapy:**\n   - **Metformin** combined with lifestyle modifications (diet, exercise).\n   - Target HbA1c: <7.0%.\n\n2. **Second-line Therapy:**\n   - **GLP-1 receptor agonist**: Preferred if the patient has atherosclerotic cardiovascular disease (ASCVD) or requires weight loss.\n   - **SGLT-2 inhibitor**: Preferred if the patient has heart failure or chronic kidney disease (CKD).\n\n3. **Third-line Therapy:**\n   - **Insulin**: Considered if HbA1c remains uncontrolled despite the above measures.\n\n### Recommendations:\n- **Ibupr

In [14]:
# Healthcare Example 2: Multi-condition multi-drug safety analysis
run_agent_and_print(
    healthcare_agent,
    "A patient has atrial fibrillation and hypertension. They are on warfarin and were just prescribed "
    "aspirin by another physician. What are the relevant interactions and what do the guidelines say?"
)

Query: A patient has atrial fibrillation and hypertension. They are on warfarin and were just prescribed aspirin by another physician. What are the relevant interactions and what do the guidelines say?
[Step 1] HumanMessage
  Content: A patient has atrial fibrillation and hypertension. They are on warfarin and were just prescribed aspirin by another physician. What are the relevant interactions and what do the guidelines say?

[Step 2] AIMessage
  Tool called: check_drug_interaction
  Arguments:   {'drug_a': 'warfarin', 'drug_b': 'aspirin'}
  Tool called: get_clinical_guideline
  Arguments:   {'condition': 'atrial_fibrillation'}
  Tool called: get_clinical_guideline
  Arguments:   {'condition': 'hypertension'}

[Step 3] ToolMessage
  Content: Interaction severity: MAJOR
Clinical note: Concomitant use significantly increases bleeding risk. Monitor INR closely and consider GI prophylaxis.

[Step 4] ToolMessage
  Content: ESC 2020 AF Guidelines: Assess stroke risk using CHA2DS2-VASc score

"### Drug Interaction\n- **Severity**: Major\n- **Clinical Note**: The combination of warfarin and aspirin significantly increases the risk of bleeding. Close monitoring of INR is essential, and gastrointestinal prophylaxis should be considered to reduce the risk of GI bleeding.\n\n### Clinical Guidelines\n#### Atrial Fibrillation (ESC 2020 Guidelines):\n- **Stroke Risk Assessment**: Use the CHA2DS2-VASc score. Anticoagulation is recommended for:\n  - Males with a score ≥ 2\n  - Females with a score ≥ 3\n- **Preferred Anticoagulants**: Non-vitamin K oral anticoagulants (NOACs) are preferred over warfarin unless contraindicated.\n\n#### Hypertension (ACC/AHA 2017 Guidelines):\n- **Blood Pressure Target**: < 130/80 mmHg.\n- **First-Line Therapy**: \n  - ACE inhibitors or ARBs\n  - Thiazide diuretics\n  - Calcium channel blockers\n- **Combination Therapy**: Avoid combining ACE inhibitors and ARBs due to the risk of adverse renal outcomes.\n\n### Clinical Considerations\nThe addition of as

Please run the above cell again if facing any error.

## Section 6 — Adding Conversational Memory

### Theory: Stateful Conversations via LangGraph Checkpointing

By default each `invoke` call is independent — the agent has no memory of previous turns. For interactive applications (a financial analyst chatting across a session, or a clinician following up on a previous query) this is unusable.

LangGraph uses **checkpointers** to persist the full message history between invocations. Every invocation is associated with a `thread_id`. All messages sharing the same `thread_id` are part of the same conversation. Starting a new `thread_id` starts a fresh conversation — this is the thread isolation mechanism that prevents user data leaking between sessions in a multi-tenant application.

`MemorySaver` is an in-process in-memory checkpointer. It is suitable for single-session use and for this lab. In production, replace it with a persistent checkpointer backed by PostgreSQL or Azure Cosmos DB.

### Step 6.1 — Create an Agent with Memory

In [15]:
from langgraph.checkpoint.memory import MemorySaver
import uuid

memory = MemorySaver()

finance_agent_mem = create_react_agent(
    model=llm,
    tools=finance_tools,
    prompt=finance_system_prompt,
    checkpointer=memory,
)

print("Finance agent with memory created.")


def chat(agent, message: str, config: dict) -> str:
    """Send a message to a stateful agent and return the final answer."""
    result = agent.invoke(
        {"messages": [HumanMessage(content=message)]},
        config=config
    )
    return result["messages"][-1].content

Finance agent with memory created.


C:\Users\agenticaiuser\AppData\Local\Temp\ipykernel_10200\3654544809.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  finance_agent_mem = create_react_agent(


In [16]:
# All three turns share the same thread_id — forming one continuous conversation.
session_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

# Turn 1: Establish context
print("TURN 1")
q1 = "What is the current price of JPM and GS?"
print(f"User:  {q1}")
r1 = chat(finance_agent_mem, q1, session_config)
print(f"Agent: {r1}\n")

TURN 1
User:  What is the current price of JPM and GS?
Agent: The current price of JPM (JPMorgan Chase & Co.) is $198.73 USD, and the current price of GS (Goldman Sachs Group Inc.) is $452.10 USD.



In [17]:
# Turn 2: 'those two stocks' is resolved from memory — no need to restate tickers
print("TURN 2")
q2 = "If I hold 300 shares of each of those two stocks, what is my total position value?"
print(f"User:  {q2}")
r2 = chat(finance_agent_mem, q2, session_config)
print(f"Agent: {r2}\n")

TURN 2
User:  If I hold 300 shares of each of those two stocks, what is my total position value?
Agent: Your portfolio breakdown is as follows:
- JPM: 300 shares x $198.73 = $59,619.00
- GS: 300 shares x $452.10 = $135,630.00

The total position value of your holdings is **$195,249.00 USD**.

### Risk Caveat:
Stock prices are subject to market fluctuations, and the value of your portfolio can change rapidly. Always consider market risks and consult a financial advisor for portfolio management.



In [18]:
# Turn 3: Rate analysis contextualised against the same portfolio
print("TURN 3")
q3 = "Given the current 2-year Treasury yield, should I be concerned about interest rate sensitivity in that portfolio?"
print(f"User:  {q3}")
r3 = chat(finance_agent_mem, q3, session_config)
print(f"Agent: {r3}")

TURN 3
User:  Given the current 2-year Treasury yield, should I be concerned about interest rate sensitivity in that portfolio?
Agent: The current 2-year Treasury yield is **4.85%**.

### Analysis:
Your portfolio, consisting of JPMorgan Chase (JPM) and Goldman Sachs (GS), is heavily weighted in financial stocks. These companies are sensitive to interest rate changes, as higher rates can impact their borrowing costs, net interest margins, and overall profitability. A 2-year Treasury yield of 4.85% suggests a relatively high short-term interest rate environment, which could:
1. Benefit banks like JPM and GS through higher net interest income.
2. Pose risks if higher rates lead to slower economic growth or increased credit defaults.

### Risk Caveat:
Interest rate sensitivity in your portfolio depends on the broader economic context and the specific balance sheets of JPM and GS. Monitor macroeconomic trends and consider diversifying if you're concerned about overexposure to financial sect

### Step 6.2 — Thread Isolation Demo

A new `thread_id` produces a completely fresh conversation with no access to the previous session.

In [19]:
new_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("NEW THREAD — no knowledge of the previous conversation")
q = "What were the two stocks I asked about earlier?"
print(f"User:  {q}")
r = chat(finance_agent_mem, q, new_config)
print(f"Agent: {r}")

NEW THREAD — no knowledge of the previous conversation
User:  What were the two stocks I asked about earlier?
Agent: I currently do not have memory of prior interactions or the stocks you asked about earlier. Could you please specify the stocks or provide their ticker symbols again?


## Section 7 — Multi-Domain Agent

### Theory: Tool Selection Across Domains

A hospital finance officer might simultaneously need clinical data and financial data. You can register all tools with a single agent and rely on GPT-4o to select the correct tool from the registered set based on the query context. The function-calling mechanism evaluates all descriptions simultaneously — you do not need to pre-route queries to domain-specific agents.

### Step 7.1 — Combined Finance and Healthcare Agent

In [20]:
all_tools = finance_tools + healthcare_tools

combined_prompt = (
    "You are a senior analyst at a healthcare investment firm with expertise in both clinical "
    "decision support and financial analysis. You have tools covering stock prices, portfolio "
    "valuation, interest rates, drug interactions, and clinical guidelines.\n\n"
    "For cross-domain questions, use multiple tools and synthesise the results. Always distinguish "
    "between financial analysis (informational, not investment advice) and clinical information "
    "(educational, not a substitute for professional medical judgment)."
)

combined_agent = create_react_agent(
    model=llm,
    tools=all_tools,
    prompt=combined_prompt,
)

print(f"Combined agent: {len(all_tools)} tools registered")
for t in all_tools:
    print(f"  {t.name}")

Combined agent: 5 tools registered
  get_stock_price
  calculate_portfolio_value
  get_interest_rate
  check_drug_interaction
  get_clinical_guideline


C:\Users\agenticaiuser\AppData\Local\Temp\ipykernel_10200\3423486316.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  combined_agent = create_react_agent(


In [21]:
# Cross-domain: healthcare investment analysis using both tool sets
run_agent_and_print(
    combined_agent,
    "I am evaluating positions in Pfizer (PFE) and Johnson & Johnson (JNJ). Give me their current prices. "
    "JNJ has a significant diabetes portfolio — what are the first-line diabetes treatment guidelines, "
    "and does the known interaction risk between metformin and ibuprofen represent any pipeline consideration?"
)

Query: I am evaluating positions in Pfizer (PFE) and Johnson & Johnson (JNJ). Give me their current prices. JNJ has a significant diabetes portfolio — what are the first-line diabetes treatment guidelines, and does the known interaction risk between metformin and ibuprofen represent any pipeline consideration?
[Step 1] HumanMessage
  Content: I am evaluating positions in Pfizer (PFE) and Johnson & Johnson (JNJ). Give me their current prices. JNJ has a significant diabetes portfolio — what are the first-line diabetes treatment guidelines, and does the known interaction risk between metformin and ibuprofen represent any pipeline considerat...

[Step 2] AIMessage
  Tool called: get_stock_price
  Arguments:   {'ticker': 'PFE'}
  Tool called: get_stock_price
  Arguments:   {'ticker': 'JNJ'}
  Tool called: get_clinical_guideline
  Arguments:   {'condition': 'type2_diabetes'}
  Tool called: check_drug_interaction
  Arguments:   {'drug_a': 'metformin', 'drug_b': 'ibuprofen'}

[Step 3] ToolMess

"### Financial Information\n- **Pfizer (PFE)**: Current price is **$29.54 USD**.\n- **Johnson & Johnson (JNJ)**: Current price is **$162.88 USD**.\n\n### Clinical Guidelines for Type 2 Diabetes\nThe **ADA 2024 Standards of Care** recommend:\n1. **First-line treatment**: Metformin combined with lifestyle modifications (target HbA1c < 7.0%).\n2. **Second-line options**:\n   - **GLP-1 receptor agonists**: Preferred for patients with atherosclerotic cardiovascular disease (ASCVD) or those needing weight loss.\n   - **SGLT-2 inhibitors**: Preferred for patients with heart failure or chronic kidney disease (CKD).\n3. **Third-line**: Insulin is introduced if HbA1c remains uncontrolled despite the above therapies.\n\n### Drug Interaction: Metformin and Ibuprofen\n- **Severity**: Moderate.\n- **Clinical Note**: NSAIDs like ibuprofen can reduce renal clearance of metformin, leading to increased plasma levels and a higher risk of lactic acidosis. Renal function should be monitored when these drug

## Section 8 — Streaming Agent Output

### Theory: Streaming Modes

LangGraph's `stream()` method supports two modes relevant to agents:

**`stream_mode="messages"`** — streams individual tokens as they arrive from the model. Used for real-time display in chat UIs so users see the response being written rather than waiting for completion.

**`stream_mode="values"`** — streams the full graph state after each node completes. Used for step-by-step audit logging, essential in regulated industries where every decision step must be recorded.

### Step 8.1 — Token Streaming

In [22]:
streaming_query = (
    "Calculate the value of a portfolio with 100 shares of AAPL and 50 shares of MSFT, "
    "and explain what the current Fed Funds rate means for technology growth stock valuations."
)

stream_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("Streaming agent response (tokens arrive incrementally):")
print("-" * 65)

for chunk, metadata in finance_agent_mem.stream(
    {"messages": [HumanMessage(content=streaming_query)]},
    config=stream_config,
    stream_mode="messages"
):
    # Print only text content chunks from the AI; skip tool call fragments
    if (
        hasattr(chunk, "content")
        and chunk.content
        and not getattr(chunk, "tool_calls", None)
        and not getattr(chunk, "tool_call_chunks", None)
    ):
        print(chunk.content, end="", flush=True)

print("\n" + "-" * 65)
print("Stream complete.")

Streaming agent response (tokens arrive incrementally):
-----------------------------------------------------------------
The current price of AAPL is $189.45 USD.The current price of MSFT is $415.20 USD.Current Fed Funds: 5.25%### Portfolio Valuation

1. **AAPL (Apple Inc.)**:  
   - Current price: $189.45 per share  
   - Shares held: 100  
   - Value = \( 189.45 \times 100 = 18,945 \, \text{USD} \)

2. **MSFT (Microsoft Corp.)**:  
   - Current price: $415.20 per share  
   - Shares held: 50  
   - Value = \( 415.20 \times 50 = 20,760 \, \text{USD} \)

**Total Portfolio Value**:  
\( 18,945 + 20,760 = 39,705 \, \text{USD} \)

---

### Fed Funds Rate and Technology Growth Stocks

- **Current Fed Funds Rate**: 5.25%  
The Fed Funds rate is the interest rate at which banks lend to each other overnight. It serves as a benchmark for other interest rates, including those for loans and bonds.

- **Impact on Technology Growth Stocks**:  
  Growth stocks, such as AAPL and MSFT, are sensitive

### Step 8.2 — Step-by-Step Trace for Audit Logging

In [23]:
trace_query = (
    "What is the current price of GS and the current SOFR rate? "
    "How do these two data points relate to Goldman Sachs's business model?"
)

trace_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("Step-by-step agent trace (stream_mode='values'):")
print("=" * 65)

for step_num, state in enumerate(
    finance_agent_mem.stream(
        {"messages": [HumanMessage(content=trace_query)]},
        config=trace_config,
        stream_mode="values"
    )
):
    latest = state["messages"][-1]
    print(f"Step {step_num + 1}: {type(latest).__name__}")
    if hasattr(latest, "tool_calls") and latest.tool_calls:
        for tc in latest.tool_calls:
            print(f"  Tool invoked: {tc['name']}({tc['args']})")
    elif hasattr(latest, "content") and latest.content:
        print(f"  Content preview: {str(latest.content)[:200]}")
    print()

Step-by-step agent trace (stream_mode='values'):
Step 1: HumanMessage
  Content preview: What is the current price of GS and the current SOFR rate? How do these two data points relate to Goldman Sachs's business model?

Step 2: AIMessage
  Tool invoked: get_stock_price({'ticker': 'GS'})
  Tool invoked: get_interest_rate({'rate_type': 'sofr'})

Step 3: ToolMessage
  Content preview: Current Sofr: 5.30%

Step 4: AIMessage
  Content preview: The current price of Goldman Sachs (GS) stock is **$452.10 USD**, and the current Secured Overnight Financing Rate (SOFR) is **5.30%**.

### Relationship to Goldman Sachs's Business Model:
1. **Stock 



## Section 9 — Inspecting the Agent Graph

### Theory: Graph Introspection

Because LangGraph models agents as directed graphs, you can inspect the compiled structure. This is valuable when building complex multi-node agents to verify that edges and routing conditions are wired as expected. It is also useful for generating architecture diagrams for compliance documentation.

In [24]:
print("Agent graph nodes:")
for node_name in finance_agent.nodes:
    print(f"  {node_name}")

print()
# draw_mermaid() requires no extra dependencies unlike draw_ascii() which needs grandalf.
# The Mermaid diagram shows the same node and edge structure in a readable text format.
print("Agent graph structure (Mermaid):")
graph_repr = finance_agent.get_graph()
print(graph_repr.draw_mermaid())

Agent graph nodes:
  __start__
  agent
  tools

Agent graph structure (Mermaid):
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Section 10 — Error Handling

### Theory: Graceful Degradation

Production agents must handle tool failures gracefully. LangGraph propagates tool exceptions as `ToolMessage` objects with error content. The model reads the error message and reasons about how to proceed — retry with different arguments, use a fallback, or inform the user.

### Step 10.1 — Unknown Ticker

In [25]:
error_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("Agent response when asked about an unknown ticker:")
print("-" * 65)
r = chat(
    finance_agent_mem,
    "What is the current price of UNKNOWNTICKER and how should I factor it into my portfolio?",
    error_config
)
print(r)

Agent response when asked about an unknown ticker:
-----------------------------------------------------------------
The ticker symbol "UNKNOWNTICKER" could not be found in the data source. This might indicate that the ticker is incorrect, the company is not publicly traded, or it is not listed on a major exchange.

If you provide the correct ticker or additional details, I can assist further. For portfolio considerations, ensure that all assets are accurately identified and valued.


In [26]:
# Healthcare agent with an unsupported condition
hc_mem = create_react_agent(
    model=llm,
    tools=healthcare_tools,
    prompt=healthcare_system_prompt,
    checkpointer=MemorySaver(),
)

hc_error_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("Agent response when condition is not in the database:")
print("-" * 65)
r = chat(
    hc_mem,
    "What is the current guideline for managing Parkinson's disease?",
    hc_error_config
)
print(r)

C:\Users\agenticaiuser\AppData\Local\Temp\ipykernel_10200\3824220578.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  hc_mem = create_react_agent(


Agent response when condition is not in the database:
-----------------------------------------------------------------
Currently, I do not have access to clinical guidelines for Parkinson's disease management. However, I recommend consulting trusted sources such as the National Institute for Health and Care Excellence (NICE), the American Academy of Neurology (AAN), or other evidence-based clinical guidelines for the most up-to-date recommendations.

Clinical decisions should always be made by qualified healthcare professionals, taking into account individual patient factors. If you need assistance with another condition or drug interaction, feel free to ask!


## Section 11 — Token Usage and Cost Tracking

### Theory: Managing Costs in Agentic Workflows

Agents consume significantly more tokens than single-turn completions. With every model call, the full conversation history is resent, tool results are injected into the context, and long reasoning chains accumulate many messages.

For a finance team running hundreds of portfolio queries per day, or a hospital running clinical queries across thousands of patient interactions, token cost is a first-class operational concern. Azure OpenAI charges per million input and output tokens. Monitoring usage enables budget controls, anomaly alerts, and prompt optimisation.

### Step 11.1 — Extract Token Usage from a Response

In [27]:
usage_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = finance_agent_mem.invoke(
    {"messages": [HumanMessage(
        content="What is the current price of AAPL and the US 10-year Treasury yield?"
    )]},
    config=usage_config
)

print("Token usage per message:")
print("-" * 55)

total_input = 0
total_output = 0

for msg in result["messages"]:
    if hasattr(msg, "response_metadata") and msg.response_metadata:
        usage = msg.response_metadata.get("token_usage", {})
        if usage:
            inp = usage.get("prompt_tokens", 0)
            out = usage.get("completion_tokens", 0)
            total_input += inp
            total_output += out
            print(f"  {type(msg).__name__:<25} input={inp:>5}  output={out:>5}")

print("-" * 55)
print(f"  {'Total':<25} input={total_input:>5}  output={total_output:>5}")
print(f"  Grand total: {total_input + total_output} tokens")

Token usage per message:
-------------------------------------------------------
  AIMessage                 input=  387  output=   55
  AIMessage                 input=  479  output=   65
-------------------------------------------------------
  Total                     input=  866  output=  120
  Grand total: 986 tokens


## Section 12 — Troubleshooting Reference

### Step 12.1 — Diagnostic Cell

Run this cell any time you encounter import errors or unexpected behaviour. It prints versions and tests all critical imports.

In [29]:
import importlib
import sys

print(f"Python: {sys.version}")
print()

for pkg in ["langchain_core", "langchain", "langchain_openai", "langgraph"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg:<30} {getattr(m, '__version__', 'unknown')}")
    except ImportError:
        print(f"{pkg:<30} NOT INSTALLED")

print()
print("Critical import tests:")
stmts = [
    "from langgraph.prebuilt import create_react_agent",
    "from langgraph.checkpoint.memory import MemorySaver",
    "from langchain_openai import AzureChatOpenAI",
    "from langchain_core.tools import tool",
    "from langchain_core.messages import HumanMessage",
]
for stmt in stmts:
    try:
        exec(stmt)
        print(f"  OK   {stmt}")
    except ImportError as e:
        print(f"  FAIL {stmt}")
        print(f"       Error: {e}")
        print(f"       Fix: %pip install langchain langchain-openai langgraph --upgrade --quiet")

Python: 3.12.1 (tags/v3.12.1:2305ca5, Dec  7 2023, 22:03:25) [MSC v.1937 64 bit (AMD64)]

langchain_core                 1.2.18
langchain                      1.2.12
langchain_openai               unknown
langgraph                      unknown

Critical import tests:
  OK   from langgraph.prebuilt import create_react_agent
  OK   from langgraph.checkpoint.memory import MemorySaver
  OK   from langchain_openai import AzureChatOpenAI
  OK   from langchain_core.tools import tool
  OK   from langchain_core.messages import HumanMessage


### 12.2 — Common Errors and Fixes

**`ImportError: cannot import name '_DirectlyInjectedToolArg'`**  
Cause: `langchain-core` is on version 0.3.x but `langgraph-prebuilt` requires 1.x.  
Fix: Run `%pip install langchain langchain-openai langgraph --upgrade --quiet` and restart the kernel. Do not pin `langchain-core` independently.

**`AuthenticationError` from Azure OpenAI**  
Check that `AZURE_OPENAI_API_KEY` and `AZURE_OPENAI_ENDPOINT` are correct. The endpoint must end with `/` and match the format `https://<resource>.openai.azure.com/`.

**`DeploymentNotFound` or HTTP 404**  
`AZURE_OPENAI_DEPLOYMENT_NAME` must match the exact deployment name in Azure AI Studio, not the underlying model name (e.g., your deployment may be named `gpt-4o-prod` even though the model is `gpt-4o`).

**`BadRequestError: api-version is not supported`**  
Update `AZURE_OPENAI_API_VERSION` in your `.env` file. Check the Azure OpenAI documentation for supported versions in your region.

**Agent loops indefinitely**  
Add a recursion limit: `config = {"configurable": {"thread_id": ...}, "recursion_limit": 10}`. The default is 25 steps.

## Section 13 — Consuming the Agent as an API

### Theory: Why Expose an Agent as an API?

A Jupyter notebook is the right environment for building and validating an agent. It is not the right environment for operating one. Real enterprise applications — a web portal where a financial analyst submits portfolio queries, a clinical decision-support widget embedded in an EHR, a Slack bot that answers fund manager questions — all need to call the agent over a network, not by executing Python cells.

Wrapping the agent in a **REST API** achieves three things:

**Decoupling:** The agent implementation (Python, LangChain, Azure OpenAI) is hidden behind an HTTP contract. The client does not need to know anything about LangGraph or tool definitions. It sends a JSON payload, it receives a JSON response.

**Multi-client access:** Any system that can make HTTP requests — a React frontend, a Power Automate flow, a mobile app, another microservice — can invoke the agent without any Python dependency.

**Operational control:** The API layer is where you add authentication (API keys, Azure AD tokens), rate limiting, request logging, usage metering, and circuit breakers. These concerns belong at the transport layer, not inside the agent loop.

### 13.1 Architecture Overview

The architecture has three layers:

**FastAPI server** — receives HTTP POST requests, deserialises the payload, invokes the agent, and returns the result as JSON. FastAPI is chosen because it has native async support (important for long-running agent turns), automatic OpenAPI documentation, and Pydantic validation.

**Agent layer** — the same `create_react_agent` instance built in earlier sections. The API layer is a thin wrapper; no agent logic is duplicated.

**Client layer** — any HTTP client. This lab demonstrates both the Python `requests` library (the programmatic consumer pattern) and a raw `curl`-equivalent using `httpx` with async support.

### 13.2 Threading in an API Context

In the API design, `thread_id` is passed by the **caller**. This gives clients full control over session continuity:

- A client that sends the same `thread_id` across multiple requests gets a multi-turn conversation with memory.
- A client that sends a new `thread_id` (or omits it, triggering auto-generation) gets a fresh stateless session.
- A multi-tenant system assigns one `thread_id` per authenticated user session, guaranteeing isolation.

In production, replace `MemorySaver` with a database-backed checkpointer so thread state survives server restarts.

### 13.3 Streaming Over HTTP

Agent turns can take 5 to 30 seconds depending on the number of tool calls. Returning a response only after the full turn completes produces an unacceptable user experience in interactive applications. The solution is **Server-Sent Events (SSE)**: the server emits newline-delimited JSON tokens as they arrive from the model, and the client renders them incrementally.

FastAPI supports SSE natively via `StreamingResponse`. The agent's `stream(stream_mode="messages")` call feeds tokens directly into the SSE stream with no buffering.

### Step 13.1 — Install FastAPI and Supporting Packages

FastAPI requires `uvicorn` (the ASGI server) and `pydantic` (request/response validation). `httpx` is used for the async HTTP client in the test cells. `nest_asyncio` is needed to run an async event loop inside a Jupyter kernel.

In [30]:
%pip install fastapi uvicorn pydantic httpx nest_asyncio --upgrade --quiet
print("FastAPI packages installed.")

Note: you may need to restart the kernel to use updated packages.
FastAPI packages installed.


### Step 13.2 — Define Request and Response Schemas

Pydantic models define the contract between callers and the API. Every field is typed and validated before the agent is invoked. A malformed request is rejected at the schema layer with a clear 422 error — it never reaches the agent loop.

The `thread_id` field is optional. When omitted, the server generates a new UUID, enabling stateless single-turn calls without the client managing session IDs.

In [31]:
from pydantic import BaseModel, Field
from typing import Optional
import uuid

class AgentRequest(BaseModel):
    """Payload sent by a client to invoke the agent."""
    query:     str            = Field(...,  description="The user's question or instruction.",
                                             min_length=1, max_length=4000)
    thread_id: Optional[str] = Field(None, description="Session ID for multi-turn memory. Omit for a fresh session.")
    domain:    Optional[str] = Field("finance",
                                     description="Agent domain: 'finance', 'healthcare', or 'combined'.")

class AgentResponse(BaseModel):
    """Payload returned by the API after the agent completes its turn."""
    answer:    str  = Field(...,  description="The agent's final answer.")
    thread_id: str  = Field(...,  description="The session ID used. Echo back to continue the conversation.")
    domain:    str  = Field(...,  description="The agent domain that handled this request.")
    steps:     int  = Field(...,  description="Number of messages in the agent's reasoning trace.")

class ErrorResponse(BaseModel):
    """Returned when the agent or server encounters an unrecoverable error."""
    error:     str  = Field(...,  description="Error message.")
    thread_id: str  = Field(...,  description="Session ID from the failed request.")

print("Schema classes defined:")
print(f"  AgentRequest  fields: {list(AgentRequest.model_fields.keys())}")
print(f"  AgentResponse fields: {list(AgentResponse.model_fields.keys())}")

Schema classes defined:
  AgentRequest  fields: ['query', 'thread_id', 'domain']
  AgentResponse fields: ['answer', 'thread_id', 'domain', 'steps']


### Step 13.3 — Build the FastAPI Application

Three endpoints are defined:

**`GET /health`** — returns server status and the list of registered agent domains. Health checks are required by load balancers and deployment pipelines (Azure App Service, AKS, and Azure Container Apps all poll this endpoint before routing traffic).

**`POST /agent/invoke`** — the primary endpoint. Receives an `AgentRequest`, routes to the correct agent instance, invokes it synchronously, and returns an `AgentResponse`. Suitable for backend-to-backend integrations where the caller can wait for the full response.

**`POST /agent/stream`** — the streaming endpoint. Returns `text/event-stream` (Server-Sent Events). Each event is a JSON object containing either a token chunk or a terminal `done` event. Suitable for frontend integrations where real-time token display is required.

In [32]:
import asyncio
import json as _json
import threading
import nest_asyncio
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

nest_asyncio.apply()  # allows asyncio.run() inside an already-running Jupyter event loop

# ── Agent registry — one instance per domain, each with its own checkpointer ──
AGENTS = {
    "finance":    create_react_agent(model=llm, tools=finance_tools,
                                     prompt=finance_system_prompt,
                                     checkpointer=MemorySaver()),
    "healthcare": create_react_agent(model=llm, tools=healthcare_tools,
                                     prompt=healthcare_system_prompt,
                                     checkpointer=MemorySaver()),
    "combined":   create_react_agent(model=llm, tools=finance_tools + healthcare_tools,
                                     prompt=combined_prompt,
                                     checkpointer=MemorySaver()),
}

app = FastAPI(
    title="LangChain Agent API",
    description="REST API for Azure OpenAI ReAct agents (finance, healthcare, combined).",
    version="1.0.0",
)


# ── Health check ──────────────────────────────────────────────────────────────
@app.get("/health")
def health():
    return {"status": "ok", "agents": list(AGENTS.keys())}


# ── Synchronous invoke endpoint ───────────────────────────────────────────────
@app.post("/agent/invoke", response_model=AgentResponse)
def agent_invoke(req: AgentRequest):
    """
    Invoke the agent and return the complete answer.
    Pass thread_id to continue an existing conversation;
    omit it (or send null) to start a fresh session.
    """
    domain = req.domain or "finance"
    if domain not in AGENTS:
        raise HTTPException(
            status_code=400,
            detail=f"Unknown domain '{domain}'. Valid values: {list(AGENTS.keys())}"
        )

    tid = req.thread_id or str(uuid.uuid4())
    config = {"configurable": {"thread_id": tid}, "recursion_limit": 15}

    try:
        result = AGENTS[domain].invoke(
            {"messages": [HumanMessage(content=req.query)]},
            config=config
        )
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))

    messages = result["messages"]
    return AgentResponse(
        answer=messages[-1].content,
        thread_id=tid,
        domain=domain,
        steps=len(messages),
    )


# ── Streaming SSE endpoint ────────────────────────────────────────────────────
@app.post("/agent/stream")
def agent_stream(req: AgentRequest):
    """
    Invoke the agent and stream tokens as Server-Sent Events.
    Each event is a JSON object: {"type": "token", "content": "..."}.
    A terminal event {"type": "done", "thread_id": "..."} signals completion.
    """
    domain = req.domain or "finance"
    if domain not in AGENTS:
        raise HTTPException(status_code=400,
                            detail=f"Unknown domain '{domain}'.")

    tid = req.thread_id or str(uuid.uuid4())
    config = {"configurable": {"thread_id": tid}, "recursion_limit": 15}
    agent  = AGENTS[domain]

    def token_generator():
        try:
            for chunk, _ in agent.stream(
                {"messages": [HumanMessage(content=req.query)]},
                config=config,
                stream_mode="messages"
            ):
                if (
                    hasattr(chunk, "content") and chunk.content
                    and not getattr(chunk, "tool_calls", None)
                    and not getattr(chunk, "tool_call_chunks", None)
                ):
                    payload = _json.dumps({"type": "token", "content": chunk.content})
                    yield f"data: {payload}\n\n"
        except Exception as exc:
            yield f"data: {_json.dumps({'type': 'error', 'content': str(exc)})}\n\n"
        finally:
            yield f"data: {_json.dumps({'type': 'done', 'thread_id': tid})}\n\n"

    return StreamingResponse(token_generator(), media_type="text/event-stream")


print("FastAPI app defined with endpoints:")
print("  GET  /health")
print("  POST /agent/invoke   — synchronous full response")
print("  POST /agent/stream   — streaming SSE tokens")
print(f"  Agent domains: {list(AGENTS.keys())}")

FastAPI app defined with endpoints:
  GET  /health
  POST /agent/invoke   — synchronous full response
  POST /agent/stream   — streaming SSE tokens
  Agent domains: ['finance', 'healthcare', 'combined']


C:\Users\agenticaiuser\AppData\Local\Temp\ipykernel_10200\2773938473.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  "finance":    create_react_agent(model=llm, tools=finance_tools,
C:\Users\agenticaiuser\AppData\Local\Temp\ipykernel_10200\2773938473.py:18: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  "healthcare": create_react_agent(model=llm, tools=healthcare_tools,
C:\Users\agenticaiuser\AppData\Local\Temp\ipykernel_10200\2773938473.py:21: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.

### Step 13.4 — Start the API Server in the Background

The server runs in a background daemon thread so the Jupyter kernel remains available for the client test cells below. In production, `uvicorn` is started as a standalone process (or containerised with Docker) rather than embedded in a notebook thread.

The server binds to `127.0.0.1:8000`. All test clients in subsequent cells target this address.

In [33]:
import uvicorn
import time
import socket

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000

def _run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host=SERVER_HOST, port=SERVER_PORT, log_level="warning")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

for _ in range(20):
    try:
        with socket.create_connection((SERVER_HOST, SERVER_PORT), timeout=0.5):
            break
    except OSError:
        time.sleep(0.5)
else:
    raise RuntimeError("uvicorn did not start within 10 seconds.")

print(f"API server running at http://{SERVER_HOST}:{SERVER_PORT}")
print("  Docs: http://127.0.0.1:8000/docs  (auto-generated OpenAPI UI)")
print("  Redoc: http://127.0.0.1:8000/redoc")

API server running at http://127.0.0.1:8000
  Docs: http://127.0.0.1:8000/docs  (auto-generated OpenAPI UI)
  Redoc: http://127.0.0.1:8000/redoc


### Step 13.5 — Health Check

The health endpoint is the first thing a deployment pipeline, load balancer, or monitoring system calls. A `200 OK` with `{"status": "ok"}` confirms the process is alive and the agent registry is populated.

In [34]:
import httpx

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000

resp = httpx.get(f"http://{SERVER_HOST}:{SERVER_PORT}/health")
print(f"Status: {resp.status_code}")
print(_json.dumps(resp.json(), indent=2))

Status: 200
{
  "status": "ok",
  "agents": [
    "finance",
    "healthcare",
    "combined"
  ]
}


### Step 13.6 — Single-Turn Invoke (Stateless)

The simplest API call pattern: no `thread_id` is provided, so the server auto-generates one. The response echoes the generated `thread_id` — the client can save it and include it in a follow-up request to continue the conversation. Without saving it, each call is fully independent.

In [35]:
# ── Finance query — stateless (no thread_id sent) ────────────────────────────
payload = {
    "query":  "What is the current price of AAPL, and how does the Fed Funds rate affect its valuation?",
    "domain": "finance"
}

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000

resp = httpx.post(
    f"http://{SERVER_HOST}:{SERVER_PORT}/agent/invoke",
    json=payload,
    timeout=60.0
)

print(f"HTTP status:  {resp.status_code}")
data = resp.json()
print(f"Thread ID:    {data['thread_id']}")
print(f"Domain:       {data['domain']}")
print(f"Steps:        {data['steps']}")
print()
print("ANSWER:")
print(data["answer"])

HTTP status:  200
Thread ID:    70e08c2b-c069-4478-acbd-1fbbcde31009
Domain:       finance
Steps:        4

ANSWER:
The current price of AAPL (Apple Inc.) is $189.45 USD.

### How the Fed Funds Rate Affects AAPL's Valuation:
1. **Discount Rate Impact**: The Fed Funds rate influences the risk-free rate, which is a key component in the discount rate used in valuation models like the Discounted Cash Flow (DCF). A higher Fed Funds rate increases the discount rate, reducing the present value of future cash flows, which can lower AAPL's valuation.

2. **Cost of Capital**: A higher Fed Funds rate increases borrowing costs for companies. If Apple relies on debt for operations or expansion, its cost of capital could rise, potentially impacting profitability and valuation.

3. **Investor Preferences**: Higher interest rates make fixed-income investments like bonds more attractive compared to equities. This could lead to reduced demand for stocks like AAPL, putting downward pressure on its price.

### Step 13.7 — Multi-Turn Conversation via API

The client explicitly manages the `thread_id`. The first request auto-generates one and the client captures it from the response. Every subsequent request sends the same `thread_id`, and the server resumes from the persisted `MemorySaver` state. The agent resolves references like "those stocks" from the thread history without the client restating them.

In [36]:
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000

BASE = f"http://{SERVER_HOST}:{SERVER_PORT}/agent/invoke"

def api_chat(query: str, domain: str = "finance", thread_id: str = None) -> dict:
    """Send one turn to the API. Returns the full response dict."""
    payload = {"query": query, "domain": domain}
    if thread_id:
        payload["thread_id"] = thread_id
    resp = httpx.post(BASE, json=payload, timeout=60.0)
    resp.raise_for_status()
    return resp.json()


# Turn 1 — no thread_id; server assigns one
print("TURN 1")
r1 = api_chat("What are the current prices of JPM and GS?")
tid = r1["thread_id"]  # capture for subsequent turns
print(f"Thread ID assigned: {tid}")
print(f"Agent: {r1['answer']}\n")

# Turn 2 — same thread_id; agent resolves 'those two banks' from memory
print("TURN 2")
r2 = api_chat("If I hold 400 shares of each of those two banks, what is my total position value?",
              thread_id=tid)
print(f"Agent: {r2['answer']}\n")

# Turn 3 — same thread_id; rate contextualised against established portfolio
print("TURN 3")
r3 = api_chat("Given the current 10-year Treasury yield, how does interest rate risk affect that portfolio?",
              thread_id=tid)
print(f"Agent: {r3['answer']}")

TURN 1
Thread ID assigned: 04648a4b-0e39-48f3-9442-edf2fdafd73b
Agent: The current prices are as follows:
- JPM (JPMorgan Chase & Co.): $198.73 USD
- GS (Goldman Sachs Group, Inc.): $452.10 USD

Please note that stock prices are subject to market fluctuations.

TURN 2
Agent: Your portfolio value is as follows:
- JPM: 400 shares × $198.73 = $79,492.00
- GS: 400 shares × $452.10 = $180,840.00

**Total Portfolio Value:** $260,332.00 USD

Risk Caveat: Market prices are volatile and can change rapidly. Ensure to monitor your portfolio regularly.

TURN 3
Agent: The current 10-year Treasury yield is 4.48%.

### Interest Rate Risk and Your Portfolio:
1. **Equity Sensitivity to Rates**:
   - Both JPMorgan Chase (JPM) and Goldman Sachs (GS) are financial institutions. Their stock performance can be sensitive to interest rate changes.
   - Rising rates (like the current 4.48%) can benefit banks as they may earn more from lending activities. However, higher rates can also slow economic growth, pot

### Step 13.8 — Healthcare Domain via API

Switching the `domain` field routes the request to the healthcare agent instance. The same API contract applies — the client does not need to know anything about which tools the healthcare agent has registered.

In [37]:
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000

hc_resp = httpx.post(
    f"http://{SERVER_HOST}:{SERVER_PORT}/agent/invoke",
    json={
        "query":  "A patient on warfarin has been prescribed aspirin. What is the interaction and what do the atrial fibrillation guidelines say?",
        "domain": "healthcare"
    },
    timeout=60.0
)

hc_data = hc_resp.json()
print(f"Domain: {hc_data['domain']}  |  Steps: {hc_data['steps']}  |  Thread: {hc_data['thread_id'][:16]}...")
print()
print(hc_data["answer"])

Domain: healthcare  |  Steps: 5  |  Thread: 2835a529-84ce-47...

### Drug Interaction: Warfarin and Aspirin
- **Interaction Severity:** Major
- **Clinical Note:** Concomitant use of warfarin and aspirin significantly increases the risk of bleeding. It is essential to monitor the patient's INR closely and consider gastrointestinal prophylaxis to reduce the risk of GI bleeding.

### Atrial Fibrillation Guidelines (ESC 2020)
- **Stroke Risk Assessment:** Use the CHA2DS2-VASc score to evaluate stroke risk.
  - Anticoagulation is recommended for:
    - Males with a score ≥ 2
    - Females with a score ≥ 3
- **Preferred Anticoagulants:** Non-vitamin K oral anticoagulants (NOACs) are preferred over warfarin unless contraindicated.

### Clinical Reminder
Decisions regarding the use of warfarin and aspirin together, as well as anticoagulation strategies for atrial fibrillation, must be made by qualified healthcare professionals, taking into account the patient's individual risk factors and clin

### Step 13.9 — Consuming the Streaming Endpoint

The streaming client opens a persistent HTTP connection and reads `data:` lines as they arrive. Each line contains a JSON object. The client accumulates token chunks until it receives a `{"type": "done"}` event, then closes the connection.

This pattern is identical to how a React frontend would consume the stream: an `EventSource` or `fetch` with a `ReadableStream` processes the same `text/event-stream` response.

In [38]:
import sys

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000

stream_payload = {
    "query":  "Calculate the value of a portfolio with 100 shares of AAPL and 75 shares of MSFT, "
              "and explain the impact of the current SOFR rate on technology stock valuations.",
    "domain": "finance"
}

print("Streaming response (tokens arrive incrementally):")
print("-" * 65)

full_response = ""
final_thread_id = None

with httpx.stream(
    "POST",
    f"http://{SERVER_HOST}:{SERVER_PORT}/agent/stream",
    json=stream_payload,
    timeout=90.0
) as response:
    for line in response.iter_lines():
        if not line.startswith("data: "):
            continue
        event = _json.loads(line[6:])  # strip leading 'data: '
        if event["type"] == "token":
            print(event["content"], end="", flush=True)
            full_response += event["content"]
        elif event["type"] == "done":
            final_thread_id = event.get("thread_id")
            break
        elif event["type"] == "error":
            print(f"\nStream error: {event['content']}")
            break

print("\n" + "-" * 65)
print(f"Stream complete. Thread ID: {final_thread_id}")
print(f"Total characters received: {len(full_response)}")

Streaming response (tokens arrive incrementally):
-----------------------------------------------------------------
The current price of AAPL is $189.45 USD.The current price of MSFT is $415.20 USD.Current Sofr: 5.30%### Portfolio Valuation
1. **AAPL (Apple Inc.)**: 100 shares at $189.45 per share.
   - Value = \( 100 \times 189.45 = 18,945 \, \text{USD} \)
2. **MSFT (Microsoft Corp.)**: 75 shares at $415.20 per share.
   - Value = \( 75 \times 415.20 = 31,140 \, \text{USD} \)

**Total Portfolio Value**:  
\( 18,945 + 31,140 = 50,085 \, \text{USD} \)

### Impact of SOFR Rate on Technology Stock Valuations
The current SOFR (Secured Overnight Financing Rate) is **5.30%**. This is a benchmark interest rate that influences borrowing costs and discount rates used in valuation models. Here's how it impacts technology stocks:

1. **Higher Discount Rates**: A higher SOFR increases the discount rate applied to future cash flows in valuation models. Since technology companies often have signific

### Step 13.10 — Schema Validation and Error Responses

FastAPI enforces the Pydantic schema before the request reaches the agent. An empty `query` or an invalid `domain` returns a `422 Unprocessable Entity` with a structured error body — the client receives actionable feedback without the agent being invoked.

This is the correct place for input validation: at the API boundary, not inside the agent loop.

In [39]:
# Test 1: Invalid domain — caught by the API layer, not the agent
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000

print("Test 1: Invalid domain value")
resp = httpx.post(
    f"http://{SERVER_HOST}:{SERVER_PORT}/agent/invoke",
    json={"query": "What is the Fed Funds rate?", "domain": "nonexistent"},
    timeout=30.0
)
print(f"  Status: {resp.status_code}")
print(f"  Body:   {resp.json()}")
print()

# Test 2: Missing required field — Pydantic raises 422 before any agent code runs
print("Test 2: Missing required 'query' field")
resp2 = httpx.post(
    f"http://{SERVER_HOST}:{SERVER_PORT}/agent/invoke",
    json={"domain": "finance"},
    timeout=30.0
)
print(f"  Status: {resp2.status_code}")
errors = resp2.json().get("detail", [])
for e in (errors if isinstance(errors, list) else [errors]):
    if isinstance(e, dict):
        print(f"  Field:  {e.get('loc', '')}  |  Msg: {e.get('msg', '')}")
    else:
        print(f"  Error:  {e}")

Test 1: Invalid domain value
  Status: 400
  Body:   {'detail': "Unknown domain 'nonexistent'. Valid values: ['finance', 'healthcare', 'combined']"}

Test 2: Missing required 'query' field
  Status: 422
  Field:  ['body', 'query']  |  Msg: Field required


### Step 13.11 — OpenAPI Schema Introspection

FastAPI auto-generates an OpenAPI 3.1 schema from the route definitions and Pydantic models. This schema can be used to auto-generate client SDKs in any language (TypeScript, C#, Java), shared with integration partners, or imported into Azure API Management to add authentication, rate limiting, and developer portal documentation without modifying the agent code.

In [40]:
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000

schema_resp = httpx.get(f"http://{SERVER_HOST}:{SERVER_PORT}/openapi.json")
schema = schema_resp.json()

print(f"API title:   {schema['info']['title']}")
print(f"API version: {schema['info']['version']}")
print()
print("Registered paths:")
for path, methods in schema["paths"].items():
    for method in methods:
        summary = methods[method].get("summary", "")
        print(f"  {method.upper():<6} {path:<30} {summary}")
print()
print("Component schemas:")
for name in schema.get("components", {}).get("schemas", {}):
    print(f"  {name}")
print()
print("OpenAPI UI available at: http://127.0.0.1:8000/docs")

API title:   LangChain Agent API
API version: 1.0.0

Registered paths:
  GET    /health                        Health
  POST   /agent/invoke                  Agent Invoke
  POST   /agent/stream                  Agent Stream

Component schemas:
  AgentRequest
  AgentResponse
  HTTPValidationError
  ValidationError

OpenAPI UI available at: http://127.0.0.1:8000/docs


## Summary

This lab covered the end-to-end construction of LLM agents using LangChain and Azure OpenAI with a deployed GPT-4o model.

**Foundations (Section 1)** — The ReAct pattern separates reasoning, action, and observation into explicit steps to prevent hallucination. The `_DirectlyInjectedToolArg` error originates from a sub-package version mismatch that the single-command installation approach resolves cleanly. Azure OpenAI provides the enterprise deployment path with data residency and compliance guarantees.

**Installation (Section 2)** — Install all LangChain packages together with `--upgrade` and no individual version pins. This is the most important operational principle for working with this ecosystem.

**Connectivity (Section 3)** — `AzureChatOpenAI` abstracts the Azure endpoint. `temperature=0` ensures deterministic tool-calling decisions.

**Tools (Section 4)** — The `@tool` decorator wraps Python functions. The docstring is what the model reads to decide when to invoke each tool. Finance tools (price lookup, portfolio valuation, rate lookup) and healthcare tools (drug interactions, clinical guidelines) illustrated how real-world data sources are wrapped. In production, replace mock implementations with live API calls.

**Agents (Section 5)** — `create_react_agent` compiles a LangGraph `StateGraph`. The agent autonomously decides when to call tools and when to produce a final answer. You do not write the loop yourself.

**Memory (Section 6)** — `MemorySaver` plus a `thread_id` enables multi-turn conversations. Thread isolation prevents context leaking between sessions. In production, replace `MemorySaver` with a persistent database-backed checkpointer.

**Multi-domain agents (Section 7)** — All tools from multiple domains can be registered with one agent. GPT-4o selects the appropriate tool based on query context without manual routing.

**Streaming and tracing (Section 8)** — `stream_mode="messages"` delivers tokens incrementally for responsive UIs. `stream_mode="values"` provides a full step-by-step audit trail for regulated environments.

**Observability, error handling, and cost (Sections 9 to 11)** — The graph structure is inspectable. The agent handles unknown inputs gracefully. Token usage is extractable from response metadata and must be monitored in production.

**Troubleshooting (Section 12)** — A diagnostic cell and error reference table provide a self-service path for the most common setup failures.

**Consuming the Agent as an API (Section 13)** — A FastAPI server wraps the agent behind three endpoints: `GET /health` for liveness checks, `POST /agent/invoke` for synchronous full-response calls, and `POST /agent/stream` for Server-Sent Event token streaming. Pydantic schemas validate requests at the API boundary before the agent is invoked. `thread_id` is caller-managed — passing the same ID across requests resumes a persistent session; omitting it starts a fresh one. The auto-generated OpenAPI schema enables client SDK generation and Azure API Management integration without modifying agent code.
